In [14]:
def is_variable(expr):
    return isinstance(expr, str) and expr in {'x', 'y', 'z'}

def is_constant(expr):
    return isinstance(expr, str) and not is_variable(expr)

def occurs_check(var, expr):
    if var == expr:
        return True
    if isinstance(expr, (list, tuple)):
        return any(occurs_check(var, arg) for arg in expr[1:])
    return False

def apply_substitution(subst, expr):
    if subst is None or subst == "NIL":
        return expr
    if is_variable(expr):
        return subst.get(expr, expr)
    if isinstance(expr, (list, tuple)):
        return [expr[0]] + [apply_substitution(subst, arg) for arg in expr[1:]]
    return expr

def compose_substitutions(s1, s2):
    if s1 == "NIL": return s2
    if s2 == "NIL": return s1
    result = s2.copy()
    for var, val in s1.items():
        result[var] = apply_substitution(s2, val)
    return result

def unify(psi1, psi2):
    if is_variable(psi1) or is_constant(psi1) or is_variable(psi2) or is_constant(psi2):
        if psi1 == psi2:
            return "NIL"
        elif is_variable(psi1):
            if occurs_check(psi1, psi2):
                return "FAILURE"
            return {psi1: psi2}
        elif is_variable(psi2):
            if occurs_check(psi2, psi1):
                return "FAILURE"
            return {psi2: psi1}
        else:
            return "FAILURE"

    if psi1[0] != psi2[0]:
        return "FAILURE"

    if len(psi1) != len(psi2):
        return "FAILURE"

    subst = "NIL"

    for i in range(1, len(psi1)):
        arg1 = apply_substitution(subst, psi1[i])
        arg2 = apply_substitution(subst, psi2[i])

        s = unify(arg1, arg2)
        if s == "FAILURE":
            return "FAILURE"

        if s != "NIL":
            subst = compose_substitutions(s, subst)

    return subst

# --- Execution Test for Notebook Problems ---
if __name__ == "__main__":
    # (a) P(A, B, B) and P(x, y, z)
    a1 = ['P', 'A', 'B', 'B']
    a2 = ['P', 'x', 'y', 'z']
    print("a:", unify(a1, a2))

    # (b) Q(y, G(A, B)) and Q(G(x, x), y)
    b1 = ['Q', 'y', ['G', 'A', 'B']]
    b2 = ['Q', ['G', 'x', 'x'], 'y']
    print("b:", unify(b1, b2))

    # (c) Older(Father(y), y) and Older(Father(x), John)
    c1 = ['Older', ['Father', 'y'], 'y']
    c2 = ['Older', ['Father', 'x'], 'John']
    print("c:", unify(c1, c2))

    # (d) Knows(Father(y), y) and Knows(x, x)
    d1 = ['Knows', ['Father', 'y'], 'y']
    d2 = ['Knows', 'x', 'x']
    print("d:", unify(d1, d2))


a: {'x': 'A', 'y': 'B', 'z': 'B'}
b: FAILURE
c: {'y': 'x', 'x': 'John'}
d: FAILURE
